# FinanceBench Sanity Check Notebook

This notebook mirrors `execute_full_pipeline` with one executable code cell per pipeline step.
Run top-to-bottom to inspect outputs and debug each stage in isolation.

In [3]:
# Step 0: setup imports and configuration
# This cell initializes all imports and creates output/cache directories.

from time import perf_counter
import json

from src.financebench_rag.config import load_config
from src.financebench_rag.dataset import build_doc_metadata_table, prepare_stage1_dataset
from src.financebench_rag.evaluation import (
    compute_correctness_judgements,
    compute_faithfulness_first_20,
    compute_page_hit_rate,
)
from src.financebench_rag.utils.io_utils import save_dataframe_csv, save_dataframe_json, save_json
from src.financebench_rag.naive_generation import run_naive_generation, sample_stage2_questions
from src.financebench_rag.rag_pipeline import RAGPipeline
from src.financebench_rag.retrieval_checks import run_retrieval_sanity_checks
from src.financebench_rag.vectorstore import (
    build_or_load_vectorstore,
    chunk_documents,
    download_required_pdfs,
    load_pdf_pages_with_metadata,
)

config = load_config()
config.results_dir.mkdir(parents=True, exist_ok=True)
config.pdf_dir.mkdir(parents=True, exist_ok=True)
config.vectorstore_dir.mkdir(parents=True, exist_ok=True)

print('Loaded config')
print(config)

Loaded config
PipelineConfig(dataset_id='PatronusAI/financebench', dataset_split='train', api_base_url='https://api.tokenfactory.nebius.com/v1', api_key='v1.CmQKHHN0YXRpY2tleS1lMDBkZW4xNGJ2aGs1YWowZ2ESIXNlcnZpY2VhY2NvdW50LWUwMHg4ZGVyazh0dHJ5cTFlNDIMCOydpM4GEPPH3eoCOgwI66C8mQcQwOPnmQJAAloDZTAw.AAAAAAAAAAEkdoZogNdkUIy462oJv3-9F7IqWb2jOKx_1MgpIoV8BcE0g41S1owg6ZSpgi9W28qsNb8GO3_ILQYFAW2-DSUF', generation_model='meta-llama/Llama-3.3-70B-Instruct', judge_model='deepseek-ai/DeepSeek-V3.2', ragas_model='deepseek-ai/DeepSeek-V3.2', embedding_model='BAAI/bge-small-en-v1.5', chunk_size=1000, chunk_overlap=150, retrieval_default_k=4, retrieval_hit_k_values=[1, 3, 5], data_dir=WindowsPath('data'), pdf_dir=WindowsPath('data/pdfs'), vectorstore_dir=WindowsPath('vectorstore'), results_dir=WindowsPath('results'))


In [4]:
# Step 1: load and filter dataset
# Creates filtered FinanceBench rows and repaired doc-link mapping.

t0 = perf_counter()
filtered_df, doc_mapping_df = prepare_stage1_dataset(
    dataset_id=config.dataset_id,
    split=config.dataset_split,
    output_dir=config.results_dir,
)

save_dataframe_csv(filtered_df, config.results_dir / 'stage1_filtered.csv')
save_dataframe_json(filtered_df, config.results_dir / 'stage1_filtered.json')
save_dataframe_csv(doc_mapping_df, config.results_dir / 'stage1_doc_mapping.csv')
save_dataframe_json(doc_mapping_df, config.results_dir / 'stage1_doc_mapping.json')

print(f'Stage 1 complete in {perf_counter() - t0:.1f}s')
print('Filtered rows:', len(filtered_df))
print('Doc mapping rows:', len(doc_mapping_df))
filtered_df[['financebench_id', 'question_type', 'doc_name', 'evidence_page_nums']].head()

Stage 1 complete in 2.7s
Filtered rows: 100
Doc mapping rows: 42


,financebench_id,question_type,doc_name,evidence_page_nums
0,financebench_id_00005,domain-relevant,CORNING_2022_10K,[59]
1,financebench_id_00070,domain-relevant,AMERICANWATERWORKS_2022_10K,"[80, 81]"
2,financebench_id_00080,domain-relevant,PAYPAL_2022_10K,[60]
3,financebench_id_00206,domain-relevant,JPMORGAN_2022_10K,[2]
4,financebench_id_00215,domain-relevant,VERIZON_2022_10K,"[22, 55]"


In [5]:
# Step 2: run naive generation on sampled questions
# Samples a small set per type and generates baseline answers without retrieval.

t0 = perf_counter()
stage2_questions = sample_stage2_questions(filtered_df, n_per_type=5)
naive_df = run_naive_generation(stage2_questions, config)

save_dataframe_csv(naive_df, config.results_dir / 'stage2_naive.csv')
save_dataframe_json(naive_df, config.results_dir / 'stage2_naive.json')

print(f'Stage 2 complete in {perf_counter() - t0:.1f}s')
print('Sampled questions:', len(stage2_questions))
naive_df.head()

Stage 2 complete in 14.5s
Sampled questions: 10


,financebench_id,question_type,question,ground_truth,naive_answer
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,Yes. Corning had a positive working capital am...,"Based on Corning's FY2022 data, I do not know ..."
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,"No, American Water Works had negative working ...",I do not know the working capital of American ...
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,Yes. Paypal has a positive working capital of ...,"Based on PayPal's FY2022 data, I do not have t..."
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"Since JPM is a financial institution, gross ma...",Gross margins are not a relevant metric for JP...
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,"Yes, Verizon is a capital-intensive business. ..."


In [8]:
# Step 3: prepare or load vectorstore
# Reuses cached FAISS if available; otherwise downloads PDFs, loads pages, chunks, and builds index.

t0 = perf_counter()
index_file = config.vectorstore_dir / 'index.faiss'
if index_file.exists():
    print(f'Reusing cached vectorstore from {config.vectorstore_dir}')
    vectorstore = build_or_load_vectorstore([], config)
    pages = []
    chunks = []
else:
    print('No cached index found. Building vectorstore from source PDFs')
    doc_table = build_doc_metadata_table(filtered_df)
    _ = download_required_pdfs(doc_table['doc_name'].tolist(), config.pdf_dir)
    pages = load_pdf_pages_with_metadata(doc_table, config.pdf_dir)
    chunks = chunk_documents(
        pages,
        chunk_size=config.chunk_size,
        chunk_overlap=config.chunk_overlap,
    )
    vectorstore = build_or_load_vectorstore(chunks, config)

print(f'Stage 3 complete in {perf_counter() - t0:.1f}s')
if chunks:
    print('Chunks created:', len(chunks))

Reusing cached vectorstore from vectorstore


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing vectorstore from vectorstore ...
Stage 3 complete in 4.5s


In [ ]:
# Step 3b: retrieval sanity checks
# Verifies retrieved pages/docs on a tiny sample before full evaluation.

t0 = perf_counter()
sanity_input = filtered_df.head(3)
sanity_df = run_retrieval_sanity_checks(sanity_input, vectorstore, k=config.retrieval_default_k)
save_dataframe_csv(sanity_df, config.results_dir / 'stage3_sanity_checks.csv')
save_dataframe_json(sanity_df, config.results_dir / 'stage3_sanity_checks.json')

print(f'Stage 3b complete in {perf_counter() - t0:.1f}s')
sanity_df.head()

In [ ]:
# Step 4: run RAG on stage2 sampled questions
# Builds RAG outputs for the same sample used by naive baseline.

t0 = perf_counter()
rag = RAGPipeline(config=config, vectorstore=vectorstore)
rag_stage2_df = rag.run_on_dataframe(stage2_questions, k=config.retrieval_default_k)
save_dataframe_csv(rag_stage2_df, config.results_dir / 'stage4_rag_stage2_questions.csv')
save_dataframe_json(rag_stage2_df, config.results_dir / 'stage4_rag_stage2_questions.json')

print(f'Stage 4 complete in {perf_counter() - t0:.1f}s')
rag_stage2_df.head()

In [ ]:
# Step 5: build side-by-side comparison
# Joins ground truth, naive output, and RAG output for quick manual inspection.

from src.financebench_rag.comparison import build_side_by_side_table

t0 = perf_counter()
comparison_df = build_side_by_side_table(stage2_questions, naive_df, rag_stage2_df)
save_dataframe_csv(comparison_df, config.results_dir / 'stage5_comparison.csv')
save_dataframe_json(comparison_df, config.results_dir / 'stage5_comparison.json')

print(f'Stage 5 complete in {perf_counter() - t0:.1f}s')
comparison_df.head(10)

In [ ]:
# Step 6a: run RAG on full filtered dataset
# Generates final RAG answers for all filtered questions.

t0 = perf_counter()
rag_full_df = rag.run_on_dataframe(filtered_df, k=config.retrieval_default_k)
save_dataframe_csv(rag_full_df, config.results_dir / 'stage6_rag_full.csv')
save_dataframe_json(rag_full_df, config.results_dir / 'stage6_rag_full.json')

print(f'Stage 6a complete in {perf_counter() - t0:.1f}s')
print('Full rows:', len(rag_full_df))

In [ ]:
# Step 6b: compute correctness
# Uses judge model to classify each RAG answer as correct/incorrect.

t0 = perf_counter()
correctness_df, correctness_score = compute_correctness_judgements(rag_full_df, config)
save_dataframe_csv(correctness_df, config.results_dir / 'stage6_correctness.csv')
save_dataframe_json(correctness_df, config.results_dir / 'stage6_correctness.json')

print(f'Stage 6b complete in {perf_counter() - t0:.1f}s')
print('Correctness accuracy:', correctness_score)
correctness_df.head()

In [ ]:
# Step 6c: compute faithfulness on first 20
# Runs ragas faithfulness scoring on a bounded subset for speed and stability.

t0 = perf_counter()
faithfulness_df, faithfulness_score = compute_faithfulness_first_20(rag_full_df, config)
save_dataframe_csv(faithfulness_df, config.results_dir / 'stage6_faithfulness.csv')
save_dataframe_json(faithfulness_df, config.results_dir / 'stage6_faithfulness.json')

print(f'Stage 6c complete in {perf_counter() - t0:.1f}s')
print('Faithfulness mean:', faithfulness_score)
faithfulness_df.head()

In [ ]:
# Step 6d: compute retrieval hit rates
# Measures evidence page overlap for configured k values.

t0 = perf_counter()
hit_detail_df, hit_summary_df = compute_page_hit_rate(
    questions_df=filtered_df,
    rag_pipeline=rag,
    k_values=config.retrieval_hit_k_values,
)
save_dataframe_csv(hit_detail_df, config.results_dir / 'stage6_hit_detail.csv')
save_dataframe_json(hit_detail_df, config.results_dir / 'stage6_hit_detail.json')
save_dataframe_csv(hit_summary_df, config.results_dir / 'stage6_hit_summary.csv')
save_dataframe_json(hit_summary_df, config.results_dir / 'stage6_hit_summary.json')

print(f'Stage 6d complete in {perf_counter() - t0:.1f}s')
hit_summary_df

In [ ]:
# Step 6e: write metrics summary
# Stores the aggregate metrics payload to JSON and prints final summary.

metrics = {
    'correctness_accuracy': correctness_score,
    'faithfulness_mean': faithfulness_score,
    'page_hit_rates': hit_summary_df.to_dict(orient='records'),
}
save_json(metrics, config.results_dir / 'stage6_metrics_summary.json')

print('Metrics summary')
print(json.dumps(metrics, indent=2))

In [ ]:
# Final bundle (matches execute_full_pipeline return shape)
results = {
    'filtered_df': filtered_df,
    'stage2_questions': stage2_questions,
    'naive_df': naive_df,
    'rag_stage2_df': rag_stage2_df,
    'comparison_df': comparison_df,
    'correctness_df': correctness_df,
    'faithfulness_df': faithfulness_df,
    'hit_detail_df': hit_detail_df,
    'hit_summary_df': hit_summary_df,
    'metrics': metrics,
}

results['metrics']